# Text Summarization using Transformers library

In [ ]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b99ea21c8a490994782eb0ff5fdae81d3a327ec2335b968b7846cbc1595f8462
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


NameError: name 'nltk' is not defined

In [ ]:
import transformers
import torch 
from transformers import AutoTokenizer 
from transformers import AutoModelForSeq2SeqLM
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import get_scheduler
import nltk
nltk.download('all')
from tqdm.auto import tqdm 
import numpy as np 

Importing and loading Google's **MT5-small model** for Text Summarization task

In [4]:
model_checkpoint = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
print(f"Model Name : {model.name_or_path}\nparameters : {model.num_parameters()}")

Model Name : google/mt5-small
parameters : 556291456


In [6]:
raw_dataset = load_dataset("abisee/cnn_dailymail","1.0.0")

README.md: 0.00B [00:00, ?B/s]

1.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

1.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

1.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

1.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

1.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

**Dataset Overview**

In [7]:
print("-"*100)
print("Dataset Overview")
print(f"Article : {raw_dataset['train']['article'][0]}")
print(f"Highlights : {raw_dataset['train']['highlights'][0]}")
print("-"*100)
print(f"{raw_dataset}")

----------------------------------------------------------------------------------------------------
Dataset Overview
Article : LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or s

**Preprocessing the dataset**

In [8]:
#renameing and removing useless columns
raw_dataset = raw_dataset.rename_column('article','text')
raw_dataset = raw_dataset.rename_column('highlights','summary')
raw_dataset = raw_dataset.remove_columns(['id'])
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['text', 'summary'],
        num_rows: 11490
    })
})

In [9]:
raw_dataset['train'] = raw_dataset['train'].select(range(10000))
raw_dataset['test'] = raw_dataset['test'].select(range(1000))
raw_dataset['validation'] = raw_dataset['validation'].select(range(1000))

In [10]:
Max_length = 512 
Max_target_length = 30

def tokenization_function(dataset):
    model_input = tokenizer(dataset['text'],max_length=Max_length,truncation=True)
    labels = tokenizer(dataset['summary'],max_length=Max_target_length,truncation=True)
    
    model_input['labels'] = labels['input_ids']
    return model_input

In [11]:
tokenized_dataset = raw_dataset.map(tokenization_function,batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(['text','summary'])
tokenized_dataset

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})

In [12]:
import evaluate 
rouge_score = evaluate.load('rouge')

In [13]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model,padding=True)

In [14]:
tokenized_dataset.set_format('torch')
Batch_size = 8
Train_dataloader = DataLoader(
    tokenized_dataset['train'],
    batch_size = Batch_size,
    shuffle=True,
    collate_fn = data_collator
)
Eval_dataloader = DataLoader(
    tokenized_dataset['validation'],
    batch_size = Batch_size,
    shuffle = False,
    collate_fn = data_collator
)

In [15]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
Epochs = 5 
num_steps_per_epochs = len(Train_dataloader)
num_train_steps = Epochs * num_steps_per_epochs
optimizer = torch.optim.AdamW(model.parameters(),lr=0.00002)
lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps=0,
    num_training_steps=num_train_steps
)

In [16]:
def postprocess_text(preds, labels):

    preds = preds.cpu().numpy()
    labels = labels.cpu().numpy()

    preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = ["\n".join(nltk.sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(nltk.sent_tokenize(label)) for label in labels]

    return preds, labels

In [18]:
print(f"Training started on {device} for {Epochs} epochs.")
progress_bar = tqdm(range(num_train_steps))

for epoch in range(Epochs):
    model.train()
    for batch in Train_dataloader:
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss 
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    

    model.eval()
    for batch in Eval_dataloader:
        batch = batch.to(device)
        with torch.no_grad():
            generated_tokens = model.generate(
                batch['input_ids'],
                attention_mask = batch['attention_mask']
            )
        labels = batch['labels']
        decoded_preds, decoded_labels = postprocess_text(generated_tokens, labels)

        rouge_score.add_batch(predictions=decoded_preds, references=decoded_labels)

    result = rouge_score.compute()
    result = {key: value * 100 for key, value in result.items()}
    result = {k: round(v, 4) for k, v in result.items()}
    print(f"[{epoch+1}/{Epochs}] | Rouge Score : {result}")   


Training started on cuda for 5 epochs.


  0%|          | 0/6250 [00:00<?, ?it/s]

[1/5] | Rouge Score : {'rouge1': np.float64(15.0872), 'rouge2': np.float64(3.6195), 'rougeL': np.float64(12.8572), 'rougeLsum': np.float64(14.0046)}
[2/5] | Rouge Score : {'rouge1': np.float64(16.0314), 'rouge2': np.float64(4.0133), 'rougeL': np.float64(13.4304), 'rougeLsum': np.float64(14.6363)}
[3/5] | Rouge Score : {'rouge1': np.float64(16.0871), 'rouge2': np.float64(4.0676), 'rougeL': np.float64(13.534), 'rougeLsum': np.float64(14.7143)}
[4/5] | Rouge Score : {'rouge1': np.float64(17.2563), 'rouge2': np.float64(4.9439), 'rougeL': np.float64(14.8611), 'rougeLsum': np.float64(15.8862)}
[5/5] | Rouge Score : {'rouge1': np.float64(17.2563), 'rouge2': np.float64(4.9439), 'rougeL': np.float64(14.8611), 'rougeLsum': np.float64(15.8862)}


# Inference 

In [25]:
text = input("Enter the paragraph for summary")
tokenized_input = tokenizer(text,return_tensors="pt").to(device)

with torch.no_grad():
    generated_tokens = model.generate(
        tokenized_input['input_ids'],
        attention_mask = tokenized_input['attention_mask']
    )

print("-"*50+"Text"+"-"*50)
print(text)
print("-"*50+"Summary"+"-"*50)
print(tokenizer.decode(generated_tokens))

--------------------------------------------------Text--------------------------------------------------
NDTV India.AI Summit LIVE Updates, Day 3: NDTV is hosting the India.AI Summit 2026 on its third day at ITC Maurya in New Delhi, beginning with a dance performance in front of an LED screen that tells the story of the Summit. As artificial intelligence transitions from experimentation to real-world deployment at scale, the Summit will explore the delicate balance between innovation and safeguards, growth and ethics, disruption and inclusion. The Summit aims to convene policymakers, technologists, entrepreneurs, and thought leaders to chart a framework for AI that is trusted, inclusive, and transformative - for India and for the world
--------------------------------------------------Summary--------------------------------------------------
['<pad> NEW: Day 3: NDTV will discuss Summit 2026 . The Summit will explore the']
